In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from prophet import Prophet

np.random.seed(42)

# -----------------------------
# 1) Create synthetic time-series dataset
# -----------------------------
start = pd.Timestamp("2025-01-01 00:00:00")
periods = 24 * 210  # 210 days of hourly data
timestamps = pd.date_range(start=start, periods=periods, freq="h")

df = pd.DataFrame({"timestamp": timestamps})
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["is_exam_period"] = (((df["timestamp"].dt.month == 4) & (df["timestamp"].dt.day >= 5)) |
                         ((df["timestamp"].dt.month == 4) & (df["timestamp"].dt.day <= 25))).astype(int)

hour_pattern = np.where((df["hour"] >= 6) & (df["hour"] <= 9), 18, 0) + \
               np.where((df["hour"] >= 18) & (df["hour"] <= 22), 25, 0)
weekend_pattern = np.where(df["is_weekend"] == 1, 12, 0)
exam_pattern = np.where(df["is_exam_period"] == 1, 20, 0)
trend = np.linspace(0, 10, len(df))
noise = np.random.normal(0, 6, len(df))

raw_usage = 35 + hour_pattern + weekend_pattern + exam_pattern + trend + noise
df["usage"] = np.clip(raw_usage, 5, None)

df["prev_hour_usage"] = df["usage"].shift(1)
df = df.dropna().reset_index(drop=True)

# Create usage categories from quantiles
q1, q2 = df["usage"].quantile([0.33, 0.66])
df["usage_category"] = pd.cut(
    df["usage"],
    bins=[-np.inf, q1, q2, np.inf],
    labels=["Low", "Medium", "High"]
)

# -----------------------------
# 2) Naive Bayes classification (time-based split)
# -----------------------------
feature_cols = ["hour", "day_of_week", "is_weekend", "is_exam_period", "prev_hour_usage"]
X = df[feature_cols]
y = df["usage_category"]

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred = nb_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
print("Naive Bayes classification metrics")
print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"]))

# -----------------------------
# 3) Prophet forecasting (daily aggregate)
# -----------------------------
daily_df = (
    df.set_index("timestamp")["usage"]
      .resample("D")
      .sum()
      .reset_index()
      .rename(columns={"timestamp": "ds", "usage": "y"})
)

holdout_days = 30
train_prophet = daily_df.iloc[:-holdout_days].copy()
test_prophet = daily_df.iloc[-holdout_days:].copy()

prophet_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.1
)
prophet_model.fit(train_prophet)

future = prophet_model.make_future_dataframe(periods=holdout_days + 45, freq="D")
forecast = prophet_model.predict(future)

# Evaluate forecast on holdout period
test_eval = test_prophet.merge(
    forecast[["ds", "yhat"]],
    on="ds",
    how="left"
)

mae = np.mean(np.abs(test_eval["y"] - test_eval["yhat"]))
rmse = np.sqrt(np.mean((test_eval["y"] - test_eval["yhat"]) ** 2))
mape = np.mean(np.abs((test_eval["y"] - test_eval["yhat"]) / test_eval["y"])) * 100

print("\nProphet holdout forecast metrics")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

# -----------------------------
# 4) What-if dashboard with timeline slider
# -----------------------------
scenarios = {
    "-15% demand": 0.85,
    "Base": 1.00,
    "+15% demand": 1.15,
    "+30% demand": 1.30,
}

future_only = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
actual_daily = daily_df.copy()

fig = go.Figure()

# Actual line
fig.add_trace(
    go.Scatter(
        x=actual_daily["ds"],
        y=actual_daily["y"],
        mode="lines",
        name="Actual Daily Usage",
        line=dict(width=2)
    )
)

# Scenario traces (only Base shown initially)
scenario_names = list(scenarios.keys())
for idx, scenario_name in enumerate(scenario_names):
    factor = scenarios[scenario_name]
    visible = True if scenario_name == "Base" else False

    fig.add_trace(
        go.Scatter(
            x=future_only["ds"],
            y=future_only["yhat"] * factor,
            mode="lines",
            name=f"Forecast ({scenario_name})",
            visible=visible,
            line=dict(dash="dash", width=3)
        )
    )

# Build slider steps for scenario selection
# Trace 0 is actual. Traces 1..N are scenarios.
steps = []
for i, scenario_name in enumerate(scenario_names):
    visibility = [True] + [False] * len(scenario_names)
    visibility[i + 1] = True

    steps.append(
        dict(
            method="update",
            label=scenario_name,
            args=[{"visible": visibility},
                  {"title": f"Hostel Laundry Usage Forecast - Scenario: {scenario_name}"}]
        )
    )

fig.update_layout(
    title="Hostel Laundry Usage Forecast - Scenario: Base",
    xaxis_title="Date",
    yaxis_title="Daily Laundry Load",
    template="plotly_white",
    sliders=[
        {
            "active": scenario_names.index("Base"),
            "currentvalue": {"prefix": "What-if scenario: "},
            "pad": {"t": 40},
            "steps": steps
        }
    ],
    xaxis={
        "rangeslider": {"visible": True},
        "type": "date"
    }
)

fig

Naive Bayes classification metrics
Accuracy: 0.4851
Macro F1: 0.4677

Classification report:
              precision    recall  f1-score   support

        High     0.7883    0.3112    0.4463       347
         Low     0.3895    0.9466    0.5519       281
      Medium     0.6117    0.3026    0.4049       380

    accuracy                         0.4851      1008
   macro avg     0.5965    0.5202    0.4677      1008
weighted avg     0.6105    0.4851    0.4601      1008

Confusion matrix (rows=true, cols=pred):
[[266  12   3]
 [239 115  26]
 [178  61 108]]

Prophet holdout forecast metrics
MAE: 42.10
RMSE: 47.82
MAPE: 3.16%
